# 07 — Earnings Reaction Study

## Why this project exists

A trader may ask: **"Do stocks react differently to earnings when volatility was already elevated before the event?"**

Free data sources do not always provide full historical earnings calendars in a robust way. For a public portfolio project, I define a small example event table manually and combine it with Yahoo Finance price data.

This demonstrates event-study thinking without pretending the dataset is institutional-grade.

In [ ]:
!pip install yfinance plotly -q

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.express as px

pd.options.display.float_format = "{:,.4f}".format

## 1. Define an example event universe

In a professional setting, this event table would come from Bloomberg, FactSet, Refinitiv, Nasdaq, or an internal earnings calendar.

In [ ]:
events = pd.DataFrame({
    "ticker": ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META"],
    "event_date": pd.to_datetime(["2025-01-30", "2025-01-29", "2025-02-26", "2025-02-06", "2025-02-04", "2025-01-29"]),
})
events

## 2. Download price data and calculate event features

I calculate:

- pre-event 10-day return
- pre-event 20-day realized volatility
- event reaction: close-to-close return from event date to next trading day

This is a simplified structure, but it shows the analytical workflow.

In [ ]:
tickers = events["ticker"].unique().tolist()
prices = yf.download(tickers, start="2024-10-01", end="2025-04-01", auto_adjust=True, progress=False)["Close"].dropna(how="all")
returns = prices.pct_change()

rows = []
for _, event in events.iterrows():
    ticker = event["ticker"]
    event_date = event["event_date"]
    px = prices[ticker].dropna()
    ret = returns[ticker].dropna()
    trading_dates = px.index
    available_dates = trading_dates[trading_dates >= event_date]
    if len(available_dates) < 2:
        continue
    d0 = available_dates[0]
    d1 = available_dates[1]
    pre_window = ret.loc[:d0].iloc[-21:-1]
    rows.append({
        "ticker": ticker,
        "event_date": d0,
        "next_trading_day": d1,
        "pre_10d_return": px.loc[d0] / px.loc[:d0].iloc[-11] - 1 if len(px.loc[:d0]) >= 11 else np.nan,
        "pre_20d_vol_ann": pre_window.std() * np.sqrt(252),
        "event_reaction_1d": px.loc[d1] / px.loc[d0] - 1,
    })

study = pd.DataFrame(rows)
study

## 3. Analyze whether pre-event volatility matters

With only a small sample, I do not overclaim. The point is to build the workflow and show how I would scale it.

In [ ]:
median_vol = study["pre_20d_vol_ann"].median()
study["vol_bucket"] = np.where(study["pre_20d_vol_ann"] >= median_vol, "High pre-event vol", "Low pre-event vol")

bucket_summary = study.groupby("vol_bucket").agg(
    avg_reaction=("event_reaction_1d", "mean"),
    median_reaction=("event_reaction_1d", "median"),
    avg_abs_reaction=("event_reaction_1d", lambda x: x.abs().mean()),
    n=("ticker", "count"),
)
bucket_summary

In [ ]:
px.scatter(study, x="pre_20d_vol_ann", y="event_reaction_1d", color="ticker", hover_data=["event_date", "pre_10d_return"], title="Earnings reaction vs pre-event realized volatility").show()
px.bar(study, x="ticker", y="event_reaction_1d", color="vol_bucket", title="1-day earnings reaction by stock").show()

## 4. My conclusion

This is a prototype event study. It demonstrates:

- Defining an event table
- Joining event dates to market data
- Building pre-event features
- Measuring post-event reaction
- Avoiding overclaiming with a tiny sample

## Next iterations

- Use a full historical earnings calendar
- Add implied volatility before earnings
- Add expected move versus realized move
- Separate beats/misses/guidance revisions
- Increase sample size across several years